In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("25.LangChain Project Basics.pdf")
documents = loader.load()

documents

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2025-12-18T00:29:43+05:30', 'author': 'Sonal Priyam', 'moddate': '2025-12-18T00:29:43+05:30', 'source': '25.LangChain Project Basics.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}, page_content='Sec 25 :- Getting Started with \nOpenAI and Ollama \n \n \n# Using Prompt Templates \n \nWe use prompt template because :-  \n \n1. To Define Roles and Behavior \nPrompt templates allow you to explicitly instruct the LLM on "how it \nshould behave or what role it should assume". \n• Instead of just asking a question, you can set a "System" \nmessage that guides the model\'s personality and expertise. \n• Example: You can instruct the model to "act as an AI engineer", \nensuring the answers are technical and precise rather than \ngeneric. \n2. To Handle Dynamic Inputs \nTemplates allow you to create flexible blueprints where specific parts \ncan be swapped out. \n• In the code exam

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=30)
docs = text_splitter.split_documents(documents)

In [8]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "HF_TOKEN"

In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-l6-v2")

C:\Users\sonal\AppData\Local\Temp\ipykernel_22584\463828749.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-l6-v2")
c:\Users\sonal\OneDrive\Documents\try2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2537.88it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-l6-v2
Key                     | Status     |  | 
------------------------+------------+-

In [14]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(documents=docs, embedding=embeddings)

In [15]:
query = "Why do we use prompt template?"
results = db.similarity_search(query)

print(results)

[Document(id='dbba6063-4fd7-4ebf-802a-a9bcef6c254b', metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2025-12-18T00:29:43+05:30', 'author': 'Sonal Priyam', 'moddate': '2025-12-18T00:29:43+05:30', 'source': '25.LangChain Project Basics.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}, page_content='2. To Handle Dynamic Inputs \nTemplates allow you to create flexible blueprints where specific parts \ncan be swapped out. \n• In the code example, the template uses {input} as a placeholder. \n• This means you can reuse the same prompt structure for \ndifferent questions without rewriting the entire instruction set \nevery time.'), Document(id='21577341-eb7c-4082-9e80-595de683a8b3', metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2025-12-18T00:29:43+05:30', 'author': 'Sonal Priyam', 'moddate': '2025-12-18T00:29:43+05:30', 'source': '25.LangChain Project Basics.pdf', 'total_pages': 18, 'pa

In [16]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b")
print(llm)

profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True} client=<groq.resources.chat.completions.Completions object at 0x000002514A987830> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002514D768DA0> model_name='openai/gpt-oss-120b' model_kwargs={} groq_api_key=SecretStr('**********')


In [17]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert. Give answer based on your good knowledge"),
    ("user","{input}")
])

prompt

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert. Give answer based on your good knowledge'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [18]:
chain = prompt | llm
response = chain.invoke({"input": "Why do we use ChatPromptTemplate?"})

print(response)

content='## TL;DR  \n**`ChatPromptTemplate`** is a helper class (most commonly seen in LangChain) that lets you **declare, compose, and render chat‑style prompts** (system‑, user‑, assistant‑messages) in a **structured, reusable, and type‑safe** way. It takes care of:\n\n| What it solves | Why it matters |\n|----------------|----------------|\n| **Message‑role ordering** (system → user → assistant) | Guarantees the LLM receives the conversation in the format it expects (e.g., OpenAI’s `ChatCompletion` API). |\n| **Variable interpolation** (`{user_name}`, `{question}`) | Lets you write one template and fill it with many different values without manual string concatenation. |\n| **Multi‑turn dialogs** (multiple messages, conditional branches) | You can model a whole “conversation flow” instead of a single static prompt. |\n| **Composable building blocks** (`+`, `append`, `extend`) | Assemble complex prompts from smaller, testable pieces. |\n| **Integration with LangChain’s chains, agents

In [19]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = prompt | llm | parser

response = response = chain.invoke({"input": "Why do we use ChatPromptTemplate?"})

print(response)

## TL;DR  
**`ChatPromptTemplate`** is a helper class (most commonly from the **LangChain** ecosystem) that lets you **declare, compose, and render structured chat‑style prompts** in a clean, reusable, and type‑safe way. It handles:

| What it solves | How `ChatPromptTemplate` helps |
|----------------|---------------------------------|
| **Multiple message roles** (system, user, assistant, …) | You can declare each role explicitly and the template will output a list of `ChatMessage` objects in the correct order. |
| **Variable interpolation** (e.g., `{name}`, `{question}`) | Variables are injected safely, with optional validation, default values, and Jinja‑style conditionals/loops. |
| **Prompt reuse & composition** | Templates can be nested, concatenated, or extended, so you can build a library of building blocks (e.g., “system intro”, “few‑shot examples”, “task instructions”). |
| **Serialization & versioning** | The template can be saved to JSON/YAML and re‑loaded, which is essenti

In [31]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """"
    Answer the following questions based on the provided context :
    <context>
    {context}
    </context>

    Question: {input}
    
    """
)

document_chain = create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='"\n    Answer the following questions based on the provided context :\n    <context>\n    {context}\n    </context>\n\n    Question: {input}\n\n    '), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002514A987830>, async_client=<groq.resources.chat.completions.AsyncC

In [32]:
db

In [33]:
retriever = db.as_retriever()

from langchain.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriever,document_chain)

retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002514D5C6F30>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='"\n    Answer the following questions based on the provided context :\n    <context>\n    {context}\n    </context>\n\n    Que

In [34]:
response = retrieval_chain.invoke({"input":"ChatPromptTemplate is used to handle dynamic inputs"})

print(response['answer'])

**Answer:** Yes.  

**Explanation:**  
`ChatPromptTemplate` (a LangChain prompt‑template class) lets you define a prompt with placeholders such as `{input}` or `{context}`. At runtime those placeholders are filled with whatever text you supply—different questions, documents, or other variables—so the same prompt structure can be reused for many different inputs without rewriting the whole instruction set. This makes it a tool specifically for handling dynamic inputs.
